# **Loading Datasets**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder


train = pd.read_csv('/content/sample_data/train_LZdllcl.csv')
test = pd.read_csv('/content/sample_data/test_2umaH9m.csv')
sample_sub = pd.read_csv('/content/sample_data/sample_submission_M0L0uXE.csv')

# **Exploratory Data Analysis**

In [2]:
# Check missing values in Train
print("--- Missing Values in Train ---")
print(train.isnull().sum()[train.isnull().sum() > 0])

# Check missing values in Test
print("\n--- Missing Values in Test ---")
print(test.isnull().sum()[test.isnull().sum() > 0])

# Check target variable class balance
print("\n--- Promotion Rate in Train ---")
print(train['is_promoted'].value_counts(normalize=True))

--- Missing Values in Train ---
education               2409
previous_year_rating    4124
dtype: int64

--- Missing Values in Test ---
education               1034
previous_year_rating    1812
dtype: int64

--- Promotion Rate in Train ---
is_promoted
0    0.91483
1    0.08517
Name: proportion, dtype: float64


# **Preprocessing & Feature Engineering**

In [3]:
def preprocess_v3(df, is_train=True, encoder=None, dept_stats=None):
    df = df.copy()

    # Missing value imputation
    df['education'] = df['education'].fillna('Unknown')
    df['previous_year_rating'] = df['previous_year_rating'].fillna(-1)

    # Core Performance Ratios
    df['total_score'] = df['avg_training_score'] * df['no_of_trainings']

    # Weighted Merit Score & High-Signal Indicator
    df['kpi_and_award'] = (
        (df['KPIs_met >80%'] * 2) +
        (df['awards_won?'] * 3) +
        ((df['previous_year_rating'] == 5.0) * 2)
    )
    df['super_star'] = ((df['KPIs_met >80%'] == 1) & (df['awards_won?'] == 1)).astype(int)

    # Age & Service Dynamics
    df['start_age'] = df['age'] - df['length_of_service']
    df['service_to_age'] = df['length_of_service'] / (df['age'] + 1e-5)

    cat_cols = ['department', 'region', 'education', 'gender', 'recruitment_channel']

    if is_train:
        encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        df[cat_cols] = encoder.fit_transform(df[cat_cols])

        # Calculating benchmark statistics on train set only
        dept_stats = {
            'dept_score': df.groupby('department')['avg_training_score'].mean().to_dict(),
            'region_score': df.groupby('region')['avg_training_score'].mean().to_dict()
        }

        df['dept_score_diff'] = df['avg_training_score'] - df['department'].map(dept_stats['dept_score'])
        df['region_score_diff'] = df['avg_training_score'] - df['region'].map(dept_stats['region_score'])
        return df, encoder, dept_stats
    else:
        df[cat_cols] = encoder.transform(df[cat_cols])
        df['dept_score_diff'] = df['avg_training_score'] - df['department'].map(dept_stats['dept_score'])
        df['region_score_diff'] = df['avg_training_score'] - df['region'].map(dept_stats['region_score'])
        return df

# Preprocessing datasets separately
train_v3, encoder, dept_stats = preprocess_v3(train, is_train=True)
test_v3 = preprocess_v3(test, is_train=False, encoder=encoder, dept_stats=dept_stats)

feature_cols = [c for c in train_v3.columns if c not in ['employee_id', 'is_promoted']]
cat_cols = ['department', 'region', 'education', 'gender', 'recruitment_channel']
cat_indices = [feature_cols.index(col) for col in cat_cols]

X = train_v3[feature_cols]
y = train_v3['is_promoted'].values
X_test = test_v3[feature_cols]

# **Model Training**

In [4]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_v3))
test_preds = np.zeros(len(test_v3))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    X_train, y_train = X.iloc[train_idx], y[train_idx]
    X_val, y_val = X.iloc[val_idx], y[val_idx]

    clf = HistGradientBoostingClassifier(
        categorical_features=cat_indices,
        max_iter=600,
        learning_rate=0.03,
        max_leaf_nodes=31,
        min_samples_leaf=20,
        l2_regularization=1.0,
        random_state=42 + fold
    )
    clf.fit(X_train, y_train)

    oof_preds[val_idx] = clf.predict_proba(X_val)[:, 1]
    test_preds += clf.predict_proba(X_test)[:, 1] / skf.n_splits

# **Fine-Tuning & Decision Threshold Optimization**

In [5]:
best_thresh = 0.50
best_f1 = 0.0

for thresh in np.arange(0.18, 0.45, 0.001):
    current_f1 = f1_score(y, (oof_preds >= thresh).astype(int))
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_thresh = thresh

print(f"10-Fold CV Optimal Threshold: {best_thresh:.3f}")
print(f"10-Fold CV Validation F1 Score: {best_f1:.4f}")

10-Fold CV Optimal Threshold: 0.285
10-Fold CV Validation F1 Score: 0.5245


In [6]:
sample_sub['is_promoted'] = (test_preds >= best_thresh).astype(int)
sample_sub.to_csv('sample4.csv', index=False)